In [35]:
from pathlib import Path
import pandas as pd

PROCESSED_DIR = Path("../data/processed")

train_contracts = pd.read_csv(PROCESSED_DIR / "train_contracts.csv")
train_clauses = pd.read_csv(PROCESSED_DIR / "train_clauses.csv")

test_contracts = pd.read_csv(PROCESSED_DIR / "test_contracts.csv")
test_clauses = pd.read_csv(PROCESSED_DIR / "test_clauses.csv")

In [36]:
print("TRAIN CONTRACTS")
print(train_contracts.shape)
print(train_contracts.columns.tolist())

print("\nTRAIN CLAUSES")
print(train_clauses.shape)
print(train_clauses.columns.tolist())

TRAIN CONTRACTS
(407, 2)
['contract_id', 'text']

TRAIN CLAUSES
(11166, 5)
['contract_id', 'category', 'clause_text', 'start', 'end']


## Chunking Strategy

The goal of this analysis is to select a chunk window size and overlap that preserve annotated CUAD clauses while keeping chunks small enough for transformer-based classification.

In [37]:
train_clauses.head()

,contract_id,category,clause_text,start,end
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Document Name,DISTRIBUTOR AGREEMENT,44,65
1,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Parties,Distributor,244,255
2,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Parties,Electric City of Illinois L.L.C.,49574,49606
3,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Parties,Electric City of Illinois LLC,212,241
4,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Parties,Company,197,204


In [38]:
train_clauses["char_length"] = (
    train_clauses["end"] - train_clauses["start"]
)

train_clauses["char_length"].describe(
    percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]
)

count    11166.000000
mean       264.172219
std        287.538126
min          1.000000
50%        197.500000
75%        368.000000
90%        591.000000
95%        758.000000
99%       1348.350000
max       3884.000000
Name: char_length, dtype: float64

In [39]:
longest_clauses = train_clauses.nlargest(
    10, "char_length"
)[
    ["category", "clause_text", "char_length"]
]

longest_clauses

,category,clause_text,char_length
7711,Source Code Escrow,Storage Fee (payable if the source code exceed...,3884
3850,Minimum Commitment,- --------------------------------------------...,3546
7708,Source Code Escrow,Subject to the provisions of Clauses 6.2 and 6...,3393
3848,Minimum Commitment,- --------------------------------------------...,3043
10413,Competitive Restriction Exception,Upon the effective date of this agreement COMW...,2820
1244,Non-Compete,The Executive agrees and undertakes with the C...,2780
4540,Renewal Term,You have the right to renew the franchise for ...,2586
9502,Revenue/Profit Sharing,In further consideration for the Service and t...,2563
5612,License Grant,"Subject to Section 3.2, a Licensor Party, on b...",2457
5616,Affiliate License-Licensor,"Subject to Section 3.2, a Licensor Party, on b...",2457


In [40]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilroberta-base")

In [41]:
train_clauses["token_length"] = train_clauses["clause_text"].apply(
    lambda text: len(
        tokenizer.encode(text, add_special_tokens=False)
    )
)

Token indices sequence length is longer than the specified maximum sequence length for this model (524 > 512). Running this sequence through the model will result in indexing errors


In [42]:
train_clauses["token_length"].describe(
    percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]
)

count    11166.000000
mean        60.776464
std         75.757942
min          1.000000
50%         41.000000
75%         82.000000
90%        136.000000
95%        184.000000
99%        345.000000
max       1989.000000
Name: token_length, dtype: float64

In [43]:
thresholds = [128, 256, 384, 510, 512]

for threshold in thresholds:
    count = (train_clauses["token_length"] > threshold).sum()
    percent = count / len(train_clauses) * 100
    
    print(
        f"Over {threshold:3} tokens: "
        f"{count:4} clauses ({percent:.2f}%)"
    )

Over 128 tokens: 1246 clauses (11.16%)
Over 256 tokens:  249 clauses (2.23%)
Over 384 tokens:   59 clauses (0.53%)
Over 510 tokens:   26 clauses (0.23%)
Over 512 tokens:   26 clauses (0.23%)


### Candidate Chunking Strategy

DistilRoBERTa supports a maximum sequence length of 512 tokens and requires
2 special tokens, leaving 510 tokens for contract text. Candidate overlap
sizes will be evaluated based on full annotation coverage and the number of
chunks generated.

In [44]:
def chunk_contract(text, window_size=510, overlap=128):
    encoding = tokenizer(
        text,
        add_special_tokens=False,
        return_offsets_mapping=True
    )

    token_ids = encoding["input_ids"]
    offsets = encoding["offset_mapping"]

    step = window_size - overlap

    chunks = []

    for token_start in range(0, len(token_ids), step):
        token_end = min(token_start + window_size, len(token_ids))

        chunk_ids = token_ids[token_start:token_end]
        chunk_offsets = offsets[token_start:token_end]

        if not chunk_ids:
            break

        char_start = chunk_offsets[0][0]
        char_end = chunk_offsets[-1][1]

        chunks.append({
            "token_start": token_start,
            "token_end": token_end,
            "char_start": char_start,
            "char_end": char_end,
            "text": text[char_start:char_end]
        })

        if token_end == len(token_ids):
            break

    return chunks

In [45]:
example_contract = train_contracts.iloc[0]

example_chunks = chunk_contract(
    example_contract["text"],
    window_size=510,
    overlap=128
)

print("Contract ID:", example_contract["contract_id"])
print("Number of chunks:", len(example_chunks))
print()

for chunk in example_chunks[:3]:
    print(
        "Tokens:",
        chunk["token_start"],
        "to",
        chunk["token_end"],
        "| Characters:",
        chunk["char_start"],
        "to",
        chunk["char_end"]
    )

Contract ID: LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
Number of chunks: 61

Tokens: 0 to 510 | Characters: 0 to 1820
Tokens: 382 to 892 | Characters: 1271 to 2689
Tokens: 764 to 1274 | Characters: 2409 to 3454


In [46]:
def evaluate_chunking(contracts, clauses, window_size, overlap):
    total_chunks = 0
    fully_covered = 0

    # Store chunks by contract ID so we only tokenize each contract once
    chunks_by_contract = {}

    for _, contract in contracts.iterrows():
        contract_id = contract["contract_id"]

        chunks = chunk_contract(
            contract["text"],
            window_size=window_size,
            overlap=overlap
        )

        chunks_by_contract[contract_id] = chunks
        total_chunks += len(chunks)

    # Check every annotated clause
    for _, clause in clauses.iterrows():
        contract_chunks = chunks_by_contract[clause["contract_id"]]

        clause_start = clause["start"]
        clause_end = clause["end"]

        covered = any(
            chunk["char_start"] <= clause_start
            and clause_end <= chunk["char_end"]
            for chunk in contract_chunks
        )

        if covered:
            fully_covered += 1

    total_clauses = len(clauses)

    return {
        "window_size": window_size,
        "overlap": overlap,
        "total_chunks": total_chunks,
        "fully_covered": fully_covered,
        "not_fully_covered": total_clauses - fully_covered,
        "coverage_percent": fully_covered / total_clauses * 100
    }

In [47]:
candidate_overlaps = [
    0, 32, 64, 96, 128,
    160, 192, 224, 256,
    288, 320, 352, 384
]

results = []

for overlap in candidate_overlaps:
    result = evaluate_chunking(
        train_contracts,
        train_clauses,
        window_size=510,
        overlap=overlap
    )
    
    results.append(result)

results_df = pd.DataFrame(results)

results_df

,window_size,overlap,total_chunks,fully_covered,not_fully_covered,coverage_percent
0,510,0,10772,9898,1268,88.644098
1,510,32,11451,10283,883,92.092065
2,510,64,12213,10617,549,95.083289
3,510,96,13127,10785,381,96.587856
4,510,128,14174,10918,248,97.778972
5,510,160,15412,10985,181,98.379008
6,510,192,16910,11024,142,98.728282
7,510,224,18716,11044,122,98.907397
8,510,256,20997,11082,84,99.247716
9,510,288,23939,11101,65,99.417876


In [48]:
def evaluate_chunking_detailed(contracts, clauses, window_size, overlap):
    total_chunks = 0
    fully_covered = 0
    too_long = 0
    boundary_failures = 0

    chunks_by_contract = {}

    for _, contract in contracts.iterrows():
        contract_id = contract["contract_id"]

        chunks = chunk_contract(
            contract["text"],
            window_size=window_size,
            overlap=overlap
        )

        chunks_by_contract[contract_id] = chunks
        total_chunks += len(chunks)

    for _, clause in clauses.iterrows():

        # This clause cannot physically fit in the selected window.
        if clause["token_length"] > window_size:
            too_long += 1
            continue

        contract_chunks = chunks_by_contract[clause["contract_id"]]

        covered = any(
            chunk["char_start"] <= clause["start"]
            and clause["end"] <= chunk["char_end"]
            for chunk in contract_chunks
        )

        if covered:
            fully_covered += 1
        else:
            boundary_failures += 1

    eligible = len(clauses) - too_long

    return {
        "overlap": overlap,
        "total_chunks": total_chunks,
        "too_long": too_long,
        "boundary_failures": boundary_failures,
        "eligible_clauses": eligible,
        "eligible_covered": fully_covered,
        "eligible_coverage_percent":
            fully_covered / eligible * 100
    }

### Refine Candidate Overlaps

The initial sweep identifies the range in which annotation coverage becomes
high. Candidate overlaps from 128 to 288 tokens are then examined in greater
detail, separating annotations that exceed the model window from failures
caused specifically by chunk boundaries.

In [49]:
candidate_overlaps = [128, 160, 192, 224, 256, 288]

detailed_results = []

for overlap in candidate_overlaps:
    detailed_results.append(
        evaluate_chunking_detailed(
            train_contracts,
            train_clauses,
            window_size=510,
            overlap=overlap
        )
    )

detailed_df = pd.DataFrame(detailed_results)

detailed_df

,overlap,total_chunks,too_long,boundary_failures,eligible_clauses,eligible_covered,eligible_coverage_percent
0,128,14174,26,222,11140,10918,98.007181
1,160,15412,26,155,11140,10985,98.608618
2,192,16910,26,116,11140,11024,98.958707
3,224,18716,26,96,11140,11044,99.138241
4,256,20997,26,58,11140,11082,99.479354
5,288,23939,26,39,11140,11101,99.649910


### Selected Chunking Parameters

We selected a **510-token text window with 256-token overlap**.

DistilRoBERTa has a maximum input length of 512 tokens and requires 2
special tokens, leaving 510 tokens available for contract text.

Clause-length analysis showed that only 26 of 11,166 training annotations
(0.23%) exceed 510 tokens and therefore cannot be fully represented within
a single model input.

Among the remaining 11,140 eligible annotations, a 256-token overlap
preserved 11,082 clauses entirely within at least one chunk, corresponding
to **99.48% eligible full-span coverage**.

Increasing overlap to 288 tokens improved eligible coverage only to 99.65%
while increasing the number of generated chunks from 20,997 to 23,939.
Therefore, 256 tokens was selected as a practical tradeoff between
annotation preservation and computational redundancy.

In [50]:
def find_boundary_failures(contracts, clauses, window_size=510, overlap=256):
    chunks_by_contract = {}

    for _, contract in contracts.iterrows():
        chunks_by_contract[contract["contract_id"]] = chunk_contract(
            contract["text"],
            window_size=window_size,
            overlap=overlap
        )

    failures = []

    for _, clause in clauses.iterrows():

        # Handle clauses that cannot physically fit separately
        if clause["token_length"] > window_size:
            continue

        chunks = chunks_by_contract[clause["contract_id"]]

        covered = any(
            chunk["char_start"] <= clause["start"]
            and clause["end"] <= chunk["char_end"]
            for chunk in chunks
        )

        if not covered:
            failures.append(clause)

    return pd.DataFrame(failures)

In [51]:
boundary_failures = find_boundary_failures(
    train_contracts,
    train_clauses
)

print("Boundary failures:", len(boundary_failures))

boundary_failures[
    ["category", "token_length", "char_length"]
].sort_values(
    "token_length",
    ascending=False
).head(20)

Boundary failures: 58


,category,token_length,char_length
765,Cap On Liability,507,1275
4230,License Grant,492,2246
4232,Non-Transferable License,492,2246
4348,Rofr/Rofo/Rofn,488,2152
7827,Revenue/Profit Sharing,479,1388
4919,Covenant Not To Sue,468,2263
3732,Exclusivity,465,1999
2567,Minimum Commitment,463,2102
10593,Anti-Assignment,455,2242
6834,Minimum Commitment,445,1271


## Map Annotation Spans onto Chunks

Each generated chunk is assigned all CUAD clause categories whose annotated
spans are fully contained within the chunk. Because multiple clause categories
may occur in the same chunk, labels are stored as multi-label category lists. A category is assigned only when the annotation is fully contained within
the chunk. Partial-span intersections are not labeled because doing so could
associate a category with a chunk that does not contain the complete
annotated clause.

In [52]:
def build_labeled_chunks(contracts, clauses, window_size=510, overlap=256):
    rows = []

    # Group annotations once so we can efficiently retrieve them by contract
    clauses_by_contract = {
        contract_id: group
        for contract_id, group in clauses.groupby("contract_id")
    }

    for _, contract in contracts.iterrows():
        contract_id = contract["contract_id"]
        text = contract["text"]

        chunks = chunk_contract(
            text,
            window_size=window_size,
            overlap=overlap
        )

        contract_clauses = clauses_by_contract.get(contract_id)

        for chunk_id, chunk in enumerate(chunks):
            labels = []

            if contract_clauses is not None:
                for _, clause in contract_clauses.iterrows():

                    # Annotation must be completely inside this chunk
                    if (
                        chunk["char_start"] <= clause["start"]
                        and clause["end"] <= chunk["char_end"]
                    ):
                        labels.append(clause["category"])

            # Remove duplicate categories within the same chunk
            labels = sorted(set(labels))

            rows.append({
    "contract_id": contract_id,
    "chunk_id": chunk_id,
    "token_start": chunk["token_start"],
    "token_end": chunk["token_end"],
    "char_start": chunk["char_start"],
    "char_end": chunk["char_end"],
    "text": chunk["text"],
    "labels": labels
})

    return pd.DataFrame(rows)

In [53]:
train_chunks = build_labeled_chunks(
    train_contracts,
    train_clauses,
    window_size=510,
    overlap=256
)

train_chunks.head()

,contract_id,chunk_id,token_start,token_end,char_start,char_end,text,labels
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,0,0,510,0,1820,EXHIBIT 10.6\n\n ...,"[Agreement Date, Document Name, Parties]"
1,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,1,254,764,735,2408,will be considered Products if Distributor e...,[Exclusivity]
2,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,2,508,1018,1819,2950,D. The Company's Appointment. The Company...,"[Exclusivity, License Grant]"
3,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,3,762,1272,2407,3452,"Market, and Distributor hereby accep...",[]
4,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,4,1016,1526,2949,3967,by such customers ...,[]


In [54]:
print("Total chunks:", len(train_chunks))

print(
    "Chunks with at least one label:",
    train_chunks["labels"].apply(len).gt(0).sum()
)

print(
    "Chunks with no labels:",
    train_chunks["labels"].apply(len).eq(0).sum()
)

print(
    "Chunks with multiple labels:",
    train_chunks["labels"].apply(len).gt(1).sum()
)

Total chunks: 20997
Chunks with at least one label: 7334
Chunks with no labels: 13663
Chunks with multiple labels: 2869


### Verify Annotation-to-Chunk Mapping

Because overlapping chunks can contain the same annotation more than once, the
number of labeled chunks is not expected to equal the number of annotations.
Instead, we verify that every eligible annotation is fully contained in at
least one generated chunk and identify any remaining boundary failures.

In [55]:
covered_annotations = 0
too_long_annotations = 0
boundary_failures = 0

for _, clause in train_clauses.iterrows():

    if clause["token_length"] > 510:
        too_long_annotations += 1
        continue

    matching_chunks = train_chunks[
        train_chunks["contract_id"] == clause["contract_id"]
    ]

    covered = (
        (matching_chunks["char_start"] <= clause["start"]) &
        (clause["end"] <= matching_chunks["char_end"])
    ).any()

    if covered:
        covered_annotations += 1
    else:
        boundary_failures += 1

print("Total annotations:", len(train_clauses))
print("Covered eligible annotations:", covered_annotations)
print("Boundary failures:", boundary_failures)
print("Too long for model:", too_long_annotations)

Total annotations: 11166
Covered eligible annotations: 11082
Boundary failures: 58
Too long for model: 26


### Validate Mapped Category Labels

The previous check confirms that eligible annotation spans are physically
contained within generated chunks. This check additionally verifies that each
contained annotation's CUAD category was actually assigned to at least one
chunk containing that full annotation span.

In [56]:
correctly_mapped = 0
mapping_failures = 0
too_long = 0
boundary_failures = 0

for _, clause in train_clauses.iterrows():

    if clause["token_length"] > 510:
        too_long += 1
        continue

    matching_chunks = train_chunks[
        train_chunks["contract_id"] == clause["contract_id"]
    ]

    containing_chunks = matching_chunks[
        (matching_chunks["char_start"] <= clause["start"]) &
        (clause["end"] <= matching_chunks["char_end"])
    ]

    if len(containing_chunks) == 0:
        boundary_failures += 1
        continue

    label_found = containing_chunks["labels"].apply(
        lambda labels: clause["category"] in labels
    ).any()

    if label_found:
        correctly_mapped += 1
    else:
        mapping_failures += 1

print("Correctly mapped:", correctly_mapped)
print("Mapping failures:", mapping_failures)
print("Boundary failures:", boundary_failures)
print("Too long:", too_long)

Correctly mapped: 11082
Mapping failures: 0
Boundary failures: 58
Too long: 26


### Analyze Uncovered Annotations

Uncovered annotations are separated into two cases:

1. **Boundary failures:** annotations containing at most 510 tokens that could
   fit within the model context but are not fully contained by any chunk.
2. **Too-long annotations:** annotations exceeding the 510-token usable model
   window and therefore cannot be represented in full within one
   DistilRoBERTa input.

The distribution of these cases across clause categories is examined before
determining how they should be handled.

In [57]:
# Recreate the DataFrame of clauses that fail because of chunk boundaries
boundary_failures_df = find_boundary_failures(
    train_contracts,
    train_clauses,
    window_size=510,
    overlap=256
)

# Clauses that are too long to fit in a 510-token text window
too_long_df = train_clauses[
    train_clauses["token_length"] > 510
].copy()

print("Boundary failures:", len(boundary_failures_df))
print("Too-long annotations:", len(too_long_df))

Boundary failures: 58
Too-long annotations: 26


In [58]:
print("BOUNDARY FAILURES BY CATEGORY")

display(
    boundary_failures_df["category"]
    .value_counts()
    .to_frame("count")
)

print("\nTOO-LONG ANNOTATIONS BY CATEGORY")

display(
    too_long_df["category"]
    .value_counts()
    .to_frame("count")
)

BOUNDARY FAILURES BY CATEGORY


,count
category,
License Grant,11
Irrevocable Or Perpetual License,6
Minimum Commitment,6
Cap On Liability,4
Non-Transferable License,4
Rofr/Rofo/Rofn,3
Liquidated Damages,3
Ip Ownership Assignment,3
Anti-Assignment,2



TOO-LONG ANNOTATIONS BY CATEGORY


,count
category,
Minimum Commitment,3
License Grant,3
Insurance,2
Non-Compete,2
Affiliate License-Licensor,2
Irrevocable Or Perpetual License,2
Source Code Escrow,2
Revenue/Profit Sharing,2
Post-Termination Services,2


## Chunking Validation Summary

The final chunking strategy uses a **510-token text window with 256-token
overlap**. This accommodates DistilRoBERTa's 512-token maximum sequence
length while reserving 2 positions for special tokens.

Across the 11,166 training annotations:

- **11,082 annotations** were fully contained in at least one generated chunk
  and correctly mapped to their CUAD category.
- **58 annotations** were short enough to fit within the model window but
  crossed the boundaries of all generated chunks.
- **26 annotations** exceeded the 510-token usable text window and therefore
  cannot be represented in full within a single DistilRoBERTa input.
- **0 mapping errors** were detected for annotations that were fully
  contained within a chunk.

Among the 11,140 annotations capable of fitting within the model window,
the selected strategy therefore achieves **99.48% full-span coverage**.

The remaining boundary and overlength cases are retained as documented
limitations rather than introducing annotation-aware chunks, which would
use ground-truth span locations unavailable during inference.

## Apply Chunking Strategy to Test Set

The chunking parameters were selected using the training data. The same
510-token window and 256-token overlap are now applied unchanged to the
held-out test contracts to avoid using test-set information to tune the
chunking strategy.

In [59]:
test_chunks = build_labeled_chunks(
    test_contracts,
    test_clauses,
    window_size=510,
    overlap=256
)

print("Train chunks:", len(train_chunks))
print("Test chunks:", len(test_chunks))

test_chunks.head()

Train chunks: 20997
Test chunks: 4283


,contract_id,chunk_id,token_start,token_end,char_start,char_end,text,labels
0,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,0,0,510,0,2241,Exhibit 10.16 SUPPLY CONTRACT Contract No: Dat...,"[Document Name, Parties]"
1,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,1,254,764,1052,3447,"""LEHEYUAN""), to purchase the products specifie...",[]
2,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,2,508,1018,2231,4407,subject to the purchase order issued by entrus...,[Insurance]
3,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,3,762,1272,3436,5495,eless paint. 7. DATE OF SHIPMENT: According to...,[Insurance]
4,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,4,1016,1526,4395,6662,Credit shall be valid until 90 days after the ...,[]


### Test Chunk Sanity Checks

The test chunks are checked for labeled, unlabeled, and multi-label examples
to confirm that the same chunk-generation process behaves as expected on
held-out contracts.

In [60]:
print("Total test chunks:", len(test_chunks))

print(
    "Test chunks with at least one label:",
    test_chunks["labels"].apply(len).gt(0).sum()
)

print(
    "Test chunks with no labels:",
    test_chunks["labels"].apply(len).eq(0).sum()
)

print(
    "Test chunks with multiple labels:",
    test_chunks["labels"].apply(len).gt(1).sum()
)

Total test chunks: 4283
Test chunks with at least one label: 1693
Test chunks with no labels: 2590
Test chunks with multiple labels: 662


### Validate Test Annotation Coverage

Test annotations are evaluated using the already-selected chunking parameters.
Annotations exceeding 510 tokens are separated from boundary failures so that
model-length limitations are distinguished from chunk-boundary limitations.

In [61]:
# Calculate token lengths for test annotations
test_clauses["token_length"] = test_clauses["clause_text"].apply(
    lambda text: len(
        tokenizer.encode(
            text,
            add_special_tokens=False
        )
    )
)

test_correctly_mapped = 0
test_mapping_failures = 0
test_boundary_failures = 0
test_too_long = 0

for _, clause in test_clauses.iterrows():

    if clause["token_length"] > 510:
        test_too_long += 1
        continue

    matching_chunks = test_chunks[
        test_chunks["contract_id"] == clause["contract_id"]
    ]

    containing_chunks = matching_chunks[
        (matching_chunks["char_start"] <= clause["start"]) &
        (clause["end"] <= matching_chunks["char_end"])
    ]

    if len(containing_chunks) == 0:
        test_boundary_failures += 1
        continue

    label_found = containing_chunks["labels"].apply(
        lambda labels: clause["category"] in labels
    ).any()

    if label_found:
        test_correctly_mapped += 1
    else:
        test_mapping_failures += 1

print("Total test annotations:", len(test_clauses))
print("Correctly mapped:", test_correctly_mapped)
print("Mapping failures:", test_mapping_failures)
print("Boundary failures:", test_boundary_failures)
print("Too long:", test_too_long)

Total test annotations: 2643
Correctly mapped: 2624
Mapping failures: 0
Boundary failures: 17
Too long: 2


## Save Processed Chunk Datasets

The validated train and test chunks are saved for downstream multi-label
classification. Category lists are serialized as JSON strings so that
multi-label assignments can be reconstructed reliably when the CSV files
are loaded.

In [62]:
import json

train_chunks_save = train_chunks.copy()
test_chunks_save = test_chunks.copy()

train_chunks_save["labels"] = train_chunks_save["labels"].apply(json.dumps)
test_chunks_save["labels"] = test_chunks_save["labels"].apply(json.dumps)

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

train_chunk_path = PROCESSED_DIR / "train_chunks.csv"
test_chunk_path = PROCESSED_DIR / "test_chunks.csv"

train_chunks_save.to_csv(train_chunk_path, index=False)
test_chunks_save.to_csv(test_chunk_path, index=False)

print("Saved:", train_chunk_path)
print("Saved:", test_chunk_path)

Saved: ../data/processed/train_chunks.csv
Saved: ../data/processed/test_chunks.csv


In [63]:
train_check = pd.read_csv(train_chunk_path)
test_check = pd.read_csv(test_chunk_path)

# Convert JSON strings back into Python lists
train_check["labels"] = train_check["labels"].apply(json.loads)
test_check["labels"] = test_check["labels"].apply(json.loads)

print("Reloaded train chunks:", len(train_check))
print("Reloaded test chunks:", len(test_check))

print("\nExample reconstructed labels:")
print(train_check["labels"].head())

Reloaded train chunks: 20997
Reloaded test chunks: 4283

Example reconstructed labels:
0    [Agreement Date, Document Name, Parties]
1                               [Exclusivity]
2                [Exclusivity, License Grant]
3                                          []
4                                          []
Name: labels, dtype: object


In [64]:
assert len(train_check) == len(train_chunks)
assert len(test_check) == len(test_chunks)

assert train_check["contract_id"].equals(
    train_chunks["contract_id"].reset_index(drop=True)
)

assert test_check["contract_id"].equals(
    test_chunks["contract_id"].reset_index(drop=True)
)

assert train_check["labels"].tolist() == train_chunks["labels"].tolist()
assert test_check["labels"].tolist() == test_chunks["labels"].tolist()

print("\nAll saved-file integrity checks passed.")


All saved-file integrity checks passed.


## Final Output

The chunking pipeline produces model-ready multi-label datasets using a
510-token window and 256-token overlap.

Final outputs:

- **20,997 training chunks** from 407 training contracts
- **4,283 test chunks** from 102 test contracts
- **99.48% eligible full-span coverage** on training annotations
- **99.36% eligible full-span coverage** on test annotations
- **0 detected category-mapping errors** in either split

The same fixed chunking strategy is applied to both training and test
contracts. Remaining boundary and overlength cases are explicitly documented
as limitations rather than handled using annotation locations that would be
unavailable on unseen contracts.

The resulting `train_chunks.csv` and `test_chunks.csv` files are ready for
downstream multi-label clause classification.